# Mengunduh Data NO2 di Kabupaten Gresik

Notebook ini mengambil data konsentrasi **NO2 (Nitrogen Dioksida)** dari Sentinel-5P melalui openEO, lalu menyimpannya sebagai file netCDF (`.nc`) dan CSV (`.csv`).

In [1]:
import openeo

In [2]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

Authenticated using refresh token.


## Area of Interest (AOI)

Polygon lokasi pengamatan di Kabupaten Gresik. Format koordinat: `[longitude, latitude]`.

In [3]:
aoi = {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {},
      "geometry": {
        "type": "Polygon",
        "coordinates": [[
          [112.6193968, -7.1514786],
          [112.6600158, -7.1514786],
          [112.6600158, -7.1927923],
          [112.619805,  -7.1927923],
          [112.6193968, -7.1514786]
        ]]
      }
    }
  ]
}

## Load Data NO2

In [4]:
s5 = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-08-24", "2026-08-24"],
    spatial_extent={
        "west": 112.6193968,
        "south": -7.1927923,
        "east": 112.6600158,
        "north": -7.1514786,
    },
    bands=["NO2"],
)

In [5]:
# Rata-rata harian, lalu rata-rata di dalam polygon AOI
s5 = s5.aggregate_temporal_period(reducer="mean", period="day")
s5 = s5.aggregate_spatial(reducer="mean", geometries=aoi)

## Jalankan Batch Job

In [6]:
job = s5.execute_batch(title="NO2 Gresik", outputfile="../data/nc/polutan_NO2_gresik.nc")

0:00:00 Job 'j-26083114371244469701fed458970ee3': send 'start'


0:00:03 Job 'j-26083114371244469701fed458970ee3': created (progress 0%)


0:00:09 Job 'j-26083114371244469701fed458970ee3': queued (progress 0%)


0:00:16 Job 'j-26083114371244469701fed458970ee3': queued (progress 0%)


0:00:24 Job 'j-26083114371244469701fed458970ee3': queued (progress 0%)


## Konversi netCDF ke CSV

In [1]:
import netCDF4
import pandas as pd

# Buka file NetCDF
ds = netCDF4.Dataset("../data/nc/polutan_NO2_gresik.nc")

# Ambil data NO2
no2 = ds.variables["NO2"][0, :]

# Ambil waktu
time = ds.variables["t"][:]

# Ubah waktu menjadi tanggal
dates = netCDF4.num2date(
    time,
    units=ds.variables["t"].units
)

# Buat tanggal lengkap selama periode penelitian
full_dates = pd.date_range(
    start="2025-08-24",
    end="2026-08-24",
    freq="D"
)

# Pasangkan tanggal dengan nilai NO2
no2_data = {
    date.strftime("%Y-%m-%d"): float(value)
    for date, value in zip(dates, no2)
}

# Buat tabel dengan semua tanggal
df = pd.DataFrame({
    "date": full_dates.strftime("%Y-%m-%d")
})

# Masukkan nilai NO2 yang tersedia
df["NO2"] = df["date"].map(no2_data)

# Simpan ke CSV
df.to_csv(
    "../data/csv/NO2_gresik_timeseries.csv",
    index=False
)

print("CSV berhasil disimpan")
print("Jumlah baris:", len(df))

CSV berhasil disimpan
Jumlah baris: 366
